# Evaluate XGBoost Submission From MLflow

This notebook shows how to evaluate a reconstructable XGBoost model submission that was logged with `log_model_submission(...)`. It downloads the `model_submission/` bundle from MLflow, reads the manifest, loads the saved XGBoost model, rebuilds the exact feature set on `shared_set_2`, scores the test split, builds portfolio weights, and runs the shared backtest.

## 1. Configure The MLflow Run To Evaluate

The run id comes from the artifact URI. For example, this URI:

`mlflow-artifacts:/1/2590f1637d834e4ea578aa81379c5c2d/artifacts/model_submission`

has run id `2590f1637d834e4ea578aa81379c5c2d`.

In [1]:
from pathlib import Path
import json

import mlflow
import pandas as pd
import xgboost as xgb

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    load_prices,
    slice_split,
    validate_prediction_frame,
    weights_from_predictions_rank_long_only,
    weights_from_predictions_risk_adjusted,
    weights_from_predictions_top_k_equal,
    write_backtest_artifacts,
)

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
tracking_uri = 'https://adams-macbook-pro.tail5ddc35.ts.net'
run_id = '2590f1637d834e4ea578aa81379c5c2d'
artifact_path = 'model_submission'
dataset_name = 'shared_set_2'
output_dir = repo_root / 'runs' / 'evaluate_xgboost_submission_from_mlflow'
output_dir.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(tracking_uri)

print('Repo root:', repo_root)
print('Tracking URI:', mlflow.get_tracking_uri())
print('Run ID:', run_id)
print('Dataset:', dataset_name)

/Users/adamthorne/.pyenv/versions/3.12.7/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
Tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
Run ID: 2590f1637d834e4ea578aa81379c5c2d
Dataset: shared_set_2


## 2. Download The Model Submission Bundle

This downloads the entire `model_submission/` directory from MLflow. The directory should contain `manifest.json`, the model file under `artifacts/`, and optionally source notebooks/code under `source/`.

In [2]:
bundle_path = Path(
    mlflow.artifacts.download_artifacts(
        run_id=run_id,
        artifact_path=artifact_path,
    )
)

print('Downloaded bundle:', bundle_path)
print('Bundle contents:')
for path in sorted(bundle_path.rglob('*')):
    print(' ', path.relative_to(bundle_path))

Downloaded bundle: /var/folders/3y/dhkqtqns7p35w8t4_svmjz780000gn/T/tmppurgi0x2/model_submission
Bundle contents:
  artifacts
  artifacts/xgboost_model.json
  manifest.json
  source
  source/xgboost_submission_workflow.ipynb


## 3. Read The Manifest

The manifest is the contract that tells the evaluator how to recreate inference: model family, target, horizon, feature order, preprocessing assumptions, model config, and artifact paths.

In [3]:
manifest_path = bundle_path / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

print(json.dumps(manifest, indent=2))

model_family = manifest['model_family']
if model_family.lower() != 'xgboost':
    raise ValueError(f'This notebook expects an xgboost submission, got {model_family!r}')

horizon = int(manifest['horizon'])
feature_names = list(manifest['feature_names'])
model_name = manifest['model_name']

print('Model name:', model_name)
print('Target:', manifest['target'])
print('Horizon:', horizon)
print('Feature count:', len(feature_names))

{
  "artifact_files": [
    "artifacts/xgboost_model.json"
  ],
  "artifact_map": {
    "xgboost_model": "artifacts/xgboost_model.json"
  },
  "feature_names": [
    "momentum_20d",
    "momentum_60d",
    "vol_20d",
    "vol_60d",
    "rsi_14",
    "price_to_sma_20d",
    "price_to_sma_50d",
    "volume_zscore_20d",
    "beta_20d_spy",
    "bollinger_z_20d",
    "excess_return_20d_vs_spy"
  ],
  "horizon": 5,
  "model_config": {
    "artifact_format": "xgboost_json",
    "library": "xgboost",
    "params": {
      "colsample_bytree": 0.8,
      "learning_rate": 0.05,
      "max_depth": 4,
      "min_child_weight": 10,
      "n_estimators": 300,
      "n_jobs": -1,
      "objective": "reg:squarederror",
      "random_state": 42,
      "subsample": 0.8,
      "verbosity": 0
    },
    "portfolio_builder": "weights_from_predictions_rank_long_only"
  },
  "model_family": "xgboost",
  "model_name": "xgboost_submission_example",
  "notes": "Example XGBoost model submission bundle.",
  "prep

## 4. Load The XGBoost Model Artifact

The submission example logs the model under the logical artifact key `xgboost_model`. If a different logical key was used, this cell falls back to the first logged model artifact.

In [4]:
artifact_map = manifest.get('artifact_map', {})
model_relative_path = artifact_map.get('xgboost_model')
if model_relative_path is None:
    model_relative_path = manifest['artifact_files'][0]

model_path = bundle_path / model_relative_path
if not model_path.exists():
    raise FileNotFoundError(f'Model artifact not found: {model_path}')

model = xgb.XGBRegressor()
model.load_model(model_path)

print('Loaded XGBoost model from:', model_path)
print('Booster rounds:', model.get_booster().num_boosted_rounds())

Loaded XGBoost model from: /var/folders/3y/dhkqtqns7p35w8t4_svmjz780000gn/T/tmppurgi0x2/model_submission/artifacts/xgboost_model.json
Booster rounds: 300


## 5. Load `shared_set_2` Prices

For this example, evaluation uses the public `shared_set_2` dataset. In official testing, this is the line you would change to a private dataset name that exists only in your local `configs/datasets.toml`.

In [5]:
spec = get_dataset_spec(dataset_name, repo_root=repo_root)
prices = load_prices(dataset_name, repo_root=repo_root)

print('Dataset display name:', spec.name)
print('Tradable tickers:', len(spec.tickers))
print('Benchmark:', spec.benchmark_ticker)
print('Price rows:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
display(prices.head())

Dataset display name: growth_tech_innovation
Tradable tickers: 26
Benchmark: SPY
Price rows: (78468, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140663,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764154,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855568,412610800
3,2014-01-07,AAPL,19.440001,19.498571,19.211430,19.287144,16.735022,317209200
4,2014-01-08,AAPL,19.243214,19.484285,19.238930,19.409286,16.841002,258529600


## 6. Rebuild The Exact Feature Set

Feature order matters. The manifest's `feature_names` list is used directly to build the feature frame and later to order the model input columns.

In [6]:
features = build_features(prices, feature_names=feature_names)

print('Feature frame:', features.shape)
print('Feature columns in model order:')
for feature in feature_names:
    print(' ', feature)
display(features.head())

Feature frame: (78468, 13)
Feature columns in model order:
  momentum_20d
  momentum_60d
  vol_20d
  vol_60d
  rsi_14
  price_to_sma_20d
  price_to_sma_50d
  volume_zscore_20d
  beta_20d_spy
  bollinger_z_20d
  excess_return_20d_vs_spy


,date,ticker,momentum_20d,momentum_60d,vol_20d,vol_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,beta_20d_spy,bollinger_z_20d,excess_return_20d_vs_spy
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Select The Evaluation Split

This example scores the configured `test` split for `shared_set_2`. Rows with missing model features are dropped before inference.

In [7]:
test_features = slice_split(features, dataset_name, 'test', repo_root=repo_root)
test_features = test_features.dropna(subset=feature_names).reset_index(drop=True)

print('Rows to score:', test_features.shape)
print('Dates:', test_features['date'].min(), '->', test_features['date'].max())
print('Tickers:', test_features['ticker'].nunique())
display(test_features.head())

Rows to score: (26070, 13)
Dates: 2022-01-03 00:00:00 -> 2025-12-31 00:00:00
Tickers: 26


,date,ticker,momentum_20d,momentum_60d,vol_20d,vol_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,beta_20d_spy,bollinger_z_20d,excess_return_20d_vs_spy
0,2022-01-03,AAPL,0.124629,0.272075,0.018847,0.015113,59.250536,0.037153,0.118654,-0.076603,1.743145,1.602632,0.067357
1,2022-01-04,AAPL,0.086983,0.259358,0.018920,0.015242,57.717787,0.019812,0.100233,-0.215006,1.760008,1.018483,0.042436
2,2022-01-05,AAPL,0.021848,0.226632,0.018653,0.015727,43.670517,-0.008367,0.067503,-0.316555,1.704584,-0.454799,0.018121
3,2022-01-06,AAPL,-0.017592,0.217236,0.018357,0.015858,49.573506,-0.024069,0.046758,-0.218865,1.687939,-1.257013,-0.017728
4,2022-01-07,AAPL,-0.013692,0.223627,0.018353,0.015831,51.744042,-0.022442,0.044799,-0.480854,1.702966,-1.138065,-0.016646


## 8. Run Model Inference

The model receives columns in exactly the order declared by `manifest['feature_names']`.

In [ ]:
X_eval = test_features[feature_names]
scores = model.predict(X_eval)

predictions = test_features[['date', 'ticker']].copy()
predictions['horizon'] = horizon
predictions['expected_return'] = scores

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

print('Predictions:', predictions.shape)
display(predictions.head())
display(predictions['expected_return'].describe().to_frame('expected_return'))

## 9. Build Portfolio Weights

This evaluator defaults to `rank_long_only`, which is the portfolio builder used by the XGBoost submission example. If the manifest records a different builder, choose the matching block below.

In [ ]:
portfolio_builder = manifest.get('model_config', {}).get('portfolio_builder', 'weights_from_predictions_rank_long_only')

if portfolio_builder == 'weights_from_predictions_rank_long_only':
    portfolio = weights_from_predictions_rank_long_only(
        predictions,
        dataset_name=dataset_name,
        strategy_name=f'{model_name}_evaluated',
    )
elif portfolio_builder == 'weights_from_predictions_top_k_equal':
    portfolio = weights_from_predictions_top_k_equal(
        predictions,
        k=int(manifest.get('model_config', {}).get('top_k', 5)),
        dataset_name=dataset_name,
        strategy_name=f'{model_name}_evaluated',
    )
elif portfolio_builder == 'weights_from_predictions_risk_adjusted':
    portfolio = weights_from_predictions_risk_adjusted(
        predictions,
        dataset_name=dataset_name,
        strategy_name=f'{model_name}_evaluated',
    )
else:
    raise ValueError(f'Unsupported portfolio builder: {portfolio_builder}')

weights = portfolio.weights
print('Portfolio builder:', portfolio_builder)
print('Weights:', weights.shape)
display(weights.head())
display(weights.mean().sort_values(ascending=False).head(10).to_frame('average_weight'))

## 10. Backtest The Evaluated Submission

The shared backtest aligns rebalance dates to the price calendar, applies transaction costs from the dataset config, compares against SPY and equal-weight benchmarks, and computes standard metrics.

In [ ]:
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
metrics = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

metrics_table = (
    pd.DataFrame([{'metric': key, 'value': value} for key, value in sorted(metrics.items())])
    .sort_values('metric')
    .reset_index(drop=True)
)

display(metrics_table)
print('Artifacts written to:', output_dir)
print('QuantStats report:', artifact_paths['quantstats_report'])

## 11. Optional Hidden Subset Evaluation

For a private weekly test, filter `test_features` to your hidden ticker subset before the inference step. The model itself does not need to know the full universe; it only sees the rows you pass into `model.predict(...)`.

In [ ]:
# Example only. Uncomment and edit privately when running a hidden subset test.
# hidden_tickers = ['AAPL', 'MSFT', 'NVDA']
# test_features = test_features.loc[test_features['ticker'].isin(hidden_tickers)].reset_index(drop=True)